# Lesson 03 - Agentic Design Patterns

In this lesson, we explore three foundational design patterns for building effective AI agents:

1. **Clear Agent Instructions** — Crafting precise, role-defining prompts that guide agent behavior
2. **Structured Output with Pydantic Models** — Ensuring agents return predictable, validated data
3. **Single Responsibility Agents** — Designing focused agents that each do one thing well

We'll apply each pattern to a **travel destination recommender** scenario, progressively building a system that can suggest destinations, check availability, and handle logistics.


## Setup


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Pattern 1: Malinaw na Mga Tagubilin para sa Ahente

Ang pinakaepektibong pattern ay ang pinakasimple rin: pagsusulat ng malinaw, detalyadong mga tagubilin para sa iyong ahente.

Ang magagandang tagubilin ay naglalarawan ng:
- **Sino** ang ahente (persona at tono)
- **Ano** ang dapat niyang gawin (hakbang-hakbang na mga responsibilidad)
- **Paano** siya dapat kumilos (mga limitasyon at istilo)

Sa ibaba, lumikha kami ng isang travel concierge agent na may malinaw na tagubilin na humuhubog sa bawat tugon na ginagawa nito.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Istrakturang Output gamit ang Pydantic Models

Kapaki-pakinabang ang malayang teksto para sa pag-uusap, ngunit kailangan ng mga downstream na sistema ng istrakturang datos.
Sa pamamagitan ng pagsasama ng **Pydantic models** sa isang **tool function**, maaari nating:

- Tukuyin ang eksaktong schema para sa output ng ahente
- Awtomatikong beripikahin ang mga tugon
- Iintegrate nang maaasahan ang mga resulta ng ahente sa lohika ng aplikasyon

Ang susi sa pagpapatupad ay ang pagpapasa ng `response_format` kapag pinapatakbo natin ang ahente. Pinipilit nito ang
modelo na magbalik ng isang beripikadong `TravelRecommendations` na object (makikita sa `response.value`)
sa halip na malayang teksto. Ang `get_destination_details` tool ay nagbabalik din ng isang typed
`DestinationRecommendation`, kaya nananatiling istrakturang buo ang datos mula simula hanggang dulo.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## Pattern 3: Mga Ahenteng May Isa Lang na Responsibilidad

Ang mga kumplikadong gawain ay nakikinabang sa paghati-hati ng trabaho sa maraming nakatutok na ahente, bawat isa ay may iisang responsibilidad:

- Isang **Eksperto sa Destinasyon** na may alam tungkol sa mga lugar at pagkakaroon
- Isang **Tagaplano ng Logistika** na humahawak sa mga flight, hotel, at itineraryo

Ito ay kahalintulad ng prinsipyo ng software engineering na *separation of concerns* — bawat ahente ay mas madaling subukan, panatilihin, at pagbutihin nang hiwalay.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Buod

Sa araling ito, inilapat namin ang tatlong agentic na disenyo ng mga pattern sa isang senaryo ng tagapagrekomenda ng paglalakbay:

| Pattern | Pangunahing Idea | Benepisyo |
|---|---|---|
| **Malinaw na Mga Tagubilin** | Itakda ang persona, mga responsibilidad, at mga limitasyon nang maaga | Konsistenteng asal ng ahente na naaayon sa brand |
| **Istrakturadong Output** | Gamitin ang mga Pydantic models bilang format ng tugon | Napatunayan, machine-readable na mga resulta |
| **Isahang Responsibilidad** | Bigyan ang bawat ahente ng isang pokus na trabaho | Mas madaling subukan, panatilihin, at pagsamahin |

Ang mga pattern na ito ay natural na nagkakasundo — maaari mong pagsamahin ang malinaw na mga tagubilin sa istrakturadong output sa loob ng isang ahenteng may isahang responsibilidad upang bumuo ng matibay, handang-gamitin na mga sistema.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
